# **RANKINGS POR FREQUÊNCIA E SIMILARIDADE**
**Nome:** Gisela Kassick e Lívia Aragão

## Importações

In [1]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import textwrap

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

## Configurações

In [2]:
ARQUIVO_ENTRADA = "materiais_normalizados_contextual.xlsx"

ARQUIVO_SAIDA = "resultados_semantica_contextual"

PASTA_SAIDA = "resultados_materiais_3"

# Embeddings
MODELO = "all-mpnet-base-v2"
TOP_N = 20
BATCH_SIZE = 64

TERMOS_ALVO = [
    "tactile sensor",
    "tactile sensors",
    "tactile sensing",
    "electronic skin",
    "electronic skins",
    "e-skin",
    "e skin",
    "artificial skin",
    "electronic artificial skin",
]

os.makedirs(PASTA_SAIDA, exist_ok=True)

## Carregar dados

In [3]:
print("CARREGANDO DADOS")

df = pd.read_excel(ARQUIVO_ENTRADA)

print("Numero de linhas:", len(df))
print("Colunas:")
print(df.columns.tolist())

CARREGANDO DADOS
Numero de linhas: 546
Colunas:
['Title', 'Year', 'Authors', 'Journal', 'DOI', 'Abstract', 'Citations', 'Type', 'Language', 'Open Access', 'PDF', 'Landing Page', 'OpenAlex ID', 'abstract_limpo', 'contexto_electrode', 'material_extraído', 'material_normalizado', 'grupo_material', 'similaridade_grupo']


## Funções auxiliares

In [4]:
def limpar_texto(texto):

    if pd.isna(texto):
        return ""

    texto = str(texto)

    # Remover espacos repetidos
    texto = re.sub(r"\s+", " ", texto)
    return texto.strip()


def normalizar_material(material):
    material = limpar_texto(material)
    return material

# Identificar coluna

In [5]:
COLUNAS_MATERIAL = ["material_extraído"]
    
coluna_material = None

for coluna in COLUNAS_MATERIAL:
    if coluna in df.columns:
        coluna_material = coluna
        break

if coluna_material is None:
    raise ValueError("Colunas encontradas:" + ", ".join(COLUNAS_MATERIAL))

print()
print("Coluna selecionada:", coluna_material)


COLUNAS_CONTEXTO = ["contexto_electrode"]
coluna_contexto = None

for coluna in COLUNAS_CONTEXTO:
    if coluna in df.columns:
        coluna_contexto = coluna
        break

if coluna_contexto is None:
    raise ValueError("Colunas esperadas:\n" + "\n".join(COLUNAS_CONTEXTO))

print()
print("Coluna de contexto:", coluna_contexto)


Coluna selecionada: material_extraído

Coluna de contexto: contexto_electrode


## Limpeza

In [6]:
df[coluna_material] = (df[coluna_material].apply(normalizar_material))
df[coluna_contexto] = (df[coluna_contexto].apply(limpar_texto))

df = df[df[coluna_material] != ""].copy()

df = df[df[coluna_contexto] != ""].copy()

df = df.reset_index(drop=True)

print()
print("Ocorrencias com material + contexto:", len(df))


Ocorrencias com material + contexto: 258


## Construir contexto

In [7]:
print()
print("CONSTRUINDO TEXTOS CONTEXTUAIS")


def construir_texto_contextual(row):
    material = row[coluna_material]
    grupo = row["material_normalizado"]
    contexto = row[coluna_contexto]
    texto = (f"Grupo semântico: {grupo}. "
        f"Material: {material}. "
        f"Contexto: {contexto}"    )
    return texto


df["texto_contextual"] = df.apply(construir_texto_contextual, axis=1)


CONSTRUINDO TEXTOS CONTEXTUAIS


## Embedding

In [8]:
print()
print("CARREGANDO MODELO DE EMBEDDINGS CONTEXTUAIS")

print("Modelo:", MODELO)
modelo = SentenceTransformer(MODELO)
print("Modelo carregado.")


print()
print("GERANDO EMBEDDINGS DOS CONTEXTOS")


textos = (df["texto_contextual"].tolist())

embeddings_contextos = modelo.encode(
    textos,
    batch_size=BATCH_SIZE,
    show_progress_bar=True,
    normalize_embeddings=True,
    convert_to_numpy=True
)

print()
print("Dimensao dos embeddings:", embeddings_contextos.shape)

print()
print("GERANDO EMBEDDINGS DOS TERMOS-ALVO")

embeddings_alvo = modelo.encode(
    TERMOS_ALVO,
    batch_size=BATCH_SIZE,
    show_progress_bar=False,
    normalize_embeddings=True,
    convert_to_numpy=True
)

print()
print("Termos utilizados:")

for termo in TERMOS_ALVO:
    print(" -", termo)


CARREGANDO MODELO DE EMBEDDINGS CONTEXTUAIS
Modelo: all-mpnet-base-v2


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Modelo carregado.

GERANDO EMBEDDINGS DOS CONTEXTOS


Batches:   0%|          | 0/5 [00:00<?, ?it/s]


Dimensao dos embeddings: (258, 768)

GERANDO EMBEDDINGS DOS TERMOS-ALVO

Termos utilizados:
 - tactile sensor
 - tactile sensors
 - tactile sensing
 - electronic skin
 - electronic skins
 - e-skin
 - e skin
 - artificial skin
 - electronic artificial skin


## Similaridade de cosseno

In [9]:
print()
print("CALCULANDO SIMILARIDADE DE COSSENO")
matriz_similaridade = cosine_similarity(embeddings_contextos, embeddings_alvo) 
print( "Dimensao da matriz:", matriz_similaridade.shape )

for i, termo in enumerate(TERMOS_ALVO):
    nome_coluna = ("sim_" + termo.lower().replace(" ", "_").replace("-", "_"))
    df[nome_coluna] = (matriz_similaridade[:, i])

# SIMILARIDADE MEDIA DA OCORRENCIA
df["similaridade_media"] = (matriz_similaridade.mean(axis=1))

# SIMILARIDADE MAXIMA DA OCORRENCIA
df["similaridade_maxima"] = (matriz_similaridade.max(axis=1))

# TERMO COM MAIOR SIMILARIDADE
indice_maior = np.argmax(matriz_similaridade, axis=1)
df["termo_maior_similaridade"] = [TERMOS_ALVO[i] for i in indice_maior]

df.to_csv(os.path.join(PASTA_SAIDA, "similaridade_por_ocorrencia.csv"), index=False, encoding="utf-8-sig")


CALCULANDO SIMILARIDADE DE COSSENO
Dimensao da matriz: (258, 9)


## Agregando por material

In [10]:
print()
print("AGREGANDO RESULTADOS POR GRUPO SEMANTICO")

if "material_normalizado" not in df.columns:
    raise ValueError"A coluna 'material_normalizado' nao foi encontrada."


# FREQUENCIA
frequencia = (df.groupby("material_normalizado")size().reset_index(name="frequencia"))
frequencia = frequencia.rename(columns={"material_normalizado": "material"})


# SIMILARIDADE MEDIA
similaridade_media = (df.groupby("material_normalizado")["similaridade_media"].mean().reset_index())
similaridade_media = similaridade_media.rename(columns={"material_normalizado": "material"})


# SIMILARIDADE MAXIMA
similaridade_maxima = (df.groupby("material_normalizado")["similaridade_maxima"].max().reset_index())
similaridade_maxima = similaridade_maxima.rename(columns={"material_normalizado": "material"})


# DESVIO-PADRAO DA SIMILARIDADE
desvio_similaridade = (df.groupby("material_normalizado")["similaridade_media"].std().reset_index())
desvio_similaridade = desvio_similaridade.rename(columns={"material_normalizado": "material", "similaridade_media": "desvio_similaridade"})


# JUNTANDO OS RESULTADOS
resultado = frequencia.merge(similaridade_media, on="material")
resultado = resultado.merge(similaridade_maxima, on="material")
resultado = resultado.merge(desvio_similaridade, on="material", how="left")


# MOSTRAR RESULTADO

print()
print("Numero de grupos semanticos:", len(resultado))

print()
print("TOP GRUPOS POR FREQUENCIA:")
print(resultado.sort_values("frequencia", ascending=False).head(30)[["material", "frequencia", "similaridade_media"]].to_string(index=False))


AGREGANDO RESULTADOS POR GRUPO SEMANTICO

Numero de grupos semanticos: 162

TOP GRUPOS POR FREQUENCIA:
                                           material  frequencia  similaridade_media
                                    silver nanowire          16            0.272468
                                              metal          10            0.374783
                                             copper           9            0.316936
                                           graphene           9            0.315310
                                             silver           9            0.318029
                                       liquid metal           7            0.341518
                                            textile           6            0.360626
                                           hydrogel           5            0.398314
                             laser-induced graphene           4            0.237798
                                   indium tin oxide     

## Rankings

In [13]:
# FREQUENCIA
resultado = resultado.sort_values("frequencia", ascending=False).reset_index(drop=True)
resultado["rank_frequencia"] = (resultado.index + 1)

# SIMILARIDADE SEMANTICA
resultado = resultado.sort_values("similaridade_media", ascending=False).reset_index(drop=True)
resultado["rank_semantico"] = (resultado.index + 1)

# DIFERENCA ENTRE OS RANKINGS
resultado["diferenca_rank"] = (resultado["rank_frequencia"] - resultado["rank_semantico"])
resultado["diferenca_rank_abs"] = (resultado["diferenca_rank"].abs())


## Salvando

In [50]:
arquivo_ranking = os.path.join(PASTA_SAIDA, "ranking_materiais_contextual.xlsx")
resultado.to_excel(arquivo_ranking, index=False)

print()
print("Ranking salvo:", arquivo_ranking)

TOP_N = 5
# TOP POR FREQUENCIA
top_frequencia = (resultado.sort_values("frequencia", ascending=False).head(TOP_N))
top_frequencia.to_csv(os.path.join(PASTA_SAIDA, "top_frequencia.csv"), index=False, encoding="utf-8-sig")

# TOP POR SIMILARIDADE
top_semantico = (resultado.sort_values("similaridade_media", ascending=False).head(TOP_N))
top_semantico.to_csv(os.path.join(PASTA_SAIDA, "top_similaridade_contextual.csv"), index=False, encoding="utf-8-sig")

print()
print("TOP MATERIAIS POR FREQUENCIA")
print(top_frequencia[["material", "frequencia", "similaridade_media"]].to_string(index=False))

print()
print("TOP MATERIAIS POR SIMILARIDADE CONTEXTUAL")
print(top_semantico[["material", "frequencia", "similaridade_media", "similaridade_maxima", "proporcao_contextos_relevantes"]].to_string(index=False))


Ranking salvo: resultados_materiais_3\ranking_materiais_contextual.xlsx

TOP MATERIAIS POR FREQUENCIA
       material  frequencia  similaridade_media
silver nanowire          16            0.272468
          metal          10            0.374783
         silver           9            0.318029
       graphene           9            0.315310
         copper           9            0.316936

TOP MATERIAIS POR SIMILARIDADE CONTEXTUAL
                                material  frequencia  similaridade_media  similaridade_maxima  proporcao_contextos_relevantes
   liquid metal nanoparticles, galinstan           1            0.464808             0.567114                             1.0
conductive silicone rubber, silver paste           2            0.456308             0.745683                             1.0
                     paper, Liquid metal           1            0.455634             0.641997                             1.0
                                 Cu tape           1          

## Top N

In [27]:
# TOP POR FREQUENCIA 
top_frequencia = (resultado .sort_values("frequencia", ascending=False).head(TOP_N))
top_frequencia.to_csv(os.path.join(PASTA_SAIDA, "top_frequencia.csv" ), index=False, encoding="utf-8-sig")

# TOP POR SIMILARIDADE 
top_semantico = (resultado .sort_values("similaridade_media", ascending=False).head(TOP_N)) 
top_semantico.to_csv(os.path.join(PASTA_SAIDA, "top_similaridade_contextual.csv"), index=False, encoding="utf-8-sig")

# TOP POR PROPORCAO DE CONTEXTOS RELEVANTES
top_relevantes = (resultado .sort_values("proporcao_contextos_relevantes", ascending=False).head(TOP_N)) 
top_relevantes.to_csv(os.path.join(PASTA_SAIDA, "top_proporcao_contextos_relevantes.csv"), index=False, encoding="utf-8-sig" )

# EXIBIR RESULTADOS 
print() 
print("TOP MATERIAIS POR FREQUENCIA") 
print(top_frequencia[["material", "frequencia", "similaridade_media"]].to_string( index=False)) 

print() 
print("TOP MATERIAIS POR SIMILARIDADE CONTEXTUAL") 
print(top_semantico[["material", "frequencia", "similaridade_media", "similaridade_maxima", "proporcao_contextos_relevantes"]].to_string( index=False ))


TOP MATERIAIS POR FREQUENCIA
              material  frequencia  similaridade_media
       silver nanowire          16            0.272468
                 metal          10            0.374783
                silver           9            0.318029
              graphene           9            0.315310
                copper           9            0.316936
          liquid metal           7            0.341518
               textile           6            0.360626
              hydrogel           5            0.398314
          silver paste           4            0.321104
laser-induced graphene           4            0.237798

TOP MATERIAIS POR SIMILARIDADE CONTEXTUAL
                                                              material  frequencia  similaridade_media  similaridade_maxima  proporcao_contextos_relevantes
                                 liquid metal nanoparticles, galinstan           1            0.464808             0.567114                             1.0
          

## Gráficos

In [61]:
# FREQUENCIA X SIMILARIDADE
plt.figure(figsize=(9, 6))
plt.scatter(resultado["frequencia"], resultado["similaridade_media"], alpha=0.8)
plt.xlabel("Frequência de menção", fontsize=20)
plt.ylabel("Similaridade semântica média", fontsize=20)
plt.yticks(fontsize=20)
plt.xticks(fontsize=20)
plt.title("Frequência de menção vs. similaridade semântica", fontsize=20)
plt.xscale("log")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(PASTA_SAIDA, "frequencia_vs_similaridade.png"), dpi=300, bbox_inches="tight")
plt.close()


# FUNCAO PARA QUEBRAR NOMES LONGOS
def quebrar_legendas(labels, largura=25):
    return ["\n".join(textwrap.wrap(str(label), width=largura)) for label in labels    ]


# TOP N POR FREQUENCIA
grafico = top_frequencia.sort_values("frequencia", ascending=True)
plt.figure(figsize=(10,6))
plt.barh(quebrar_legendas(grafico["material"], largura=25), grafico["frequencia"])
plt.yticks(fontsize=20)
plt.xticks(fontsize=20)
plt.xlabel("Número de menções", fontsize=20)
plt.ylabel("Material", fontsize=20)
plt.title(f"Top {TOP_N} materiais por frequência", fontsize=20)
plt.tight_layout()
plt.savefig(os.path.join(PASTA_SAIDA, "top_frequencia.png"), dpi=300, bbox_inches="tight")
plt.close()


# TOP N POR SIMILARIDADE
grafico = top_semantico.sort_values("similaridade_media", ascending=True)
plt.figure(figsize=(10, 5))
plt.barh(quebrar_legendas(grafico["material"], largura=25), grafico["similaridade_media"])
plt.yticks(fontsize=20)
plt.xticks(fontsize=20)
plt.xlabel("Similaridade semântica media", fontsize=20)
plt.ylabel("Material", fontsize=20)
plt.title(f"Top {TOP_N} materiais por similaridade semântica", fontsize=20)
plt.tight_layout()
plt.savefig(os.path.join(PASTA_SAIDA, "top_similaridade.png"), dpi=300, bbox_inches="tight")
plt.close()

## Análise estatística

In [56]:
# 1. Estatísticas gerais das ocorrências
total_ocorrencias = len(df)
sim_media = df["similaridade_media"].mean()
sim_min = df["similaridade_media"].min()
sim_max = df["similaridade_media"].max()

# 2. Contagem e ordenação dos termos-alvo mais próximos
contagem_termos = df["termo_maior_similaridade"].value_counts()
top_termos = [f"\\textit{{{termo}}} ({qtd})" for termo, qtd in contagem_termos.head(4).items()]

# Formata a lista para o português (ex: "A, B, C e D")
if len(top_termos) > 1:
    string_termos = ", ".join(top_termos[:-1]) + f" e {top_termos[-1]}"
else:
    string_termos = top_termos[0]

# 3. Faixa de similaridade para materiais com 1 ocorrência
mat_freq_1 = resultado[resultado["frequencia"] == 1]
min_freq_1 = mat_freq_1["similaridade_media"].min()
max_freq_1 = mat_freq_1["similaridade_media"].max()

# 4. Análise de materiais com mais de 1 ocorrência
mat_multiplas = resultado[resultado["frequencia"] > 1]
total_multiplas = len(mat_multiplas)
min_multiplas = mat_multiplas["similaridade_media"].min()
max_multiplas = mat_multiplas["similaridade_media"].max()

# Materiais com >1 ocorrência e similaridade >= 0.40
mat_destaque = mat_multiplas[mat_multiplas["similaridade_media"] >= 0.40]["material"].tolist()
string_mat_destaque = ", ".join([f"\\textit{{{m}}}" for m in mat_destaque]) if mat_destaque else "nenhum material"

# 5. Montagem dinâmica do parágrafo
paragrafo = (
    f"Total de ocorrências: {total_ocorrencias}.\n"
    f"Similaridade média: {sim_media:.2f}\n".replace('.', ',') 
    f"Similaridade mínima: {sim_min:.2f}\n".replace('.', ',')
    f"Similaridade máxima: {sim_max:.2f}.\n".replace('.', ',')
    f"Termos mais próximos foram {string_termos}\n"
    f"Faixa de similaridade de materiais com uma única ocorrência: ({min_freq_1:.2f}–{max_freq_1:.2f})\n".replace('.', ',') +
    f"Faixa de similaridade de materiais mais frequentes:{min_multiplas:.2f} e {max_multiplas:.2f}.\n".replace('.', ',') +
    f"Entre os {total_multiplas} materiais com mais de uma ocorrência, apenas "
    f"{string_mat_destaque} apresentou similaridade média $ \\geqslant 0,40$."
)

print("PARÁGRAFO GERADO PARA O ARTIGO:\n")
print(paragrafo)

PARÁGRAFO GERADO PARA O ARTIGO:

Foram analisadas 258 ocorrências, A similaridade média foi 0,32, variando de 0,09 a aproximadamente 0,50,
Os termos mais próximos foram \textit{electronic artificial skin} (84), \textit{tactile sensors} (71), \textit{tactile sensor} (50) e \textit{electronic skin} (46), mostrando que os contextos analisados abrangem tanto sensores táteis quanto pele eletrônica,
Materiais com uma ocorrência cobriram quase toda a faixa de similaridade (0,09–0,46), enquanto os mais frequentes se concentraram aproximadamente entre 0,24 e 0,46,
Entre os 28 materiais com mais de uma ocorrência, apenas \textit{conductive silicone rubber, silver paste} apresentou similaridade média $ \geqslant 0,40$.
